In [2]:
from abc import ABC, abstractmethod
from typing import List

# 1. ABSTRACT BASE CLASS & ENCAPSULATION
class TiketBioskop(ABC):
    def __init__(self, id_tiket: str, judul_film: str, kursi: str, harga_dasar: float):
        # Encapsulation: Atribut private tidak bisa diakses langsung dari luar kelas
        self.__id_tiket = id_tiket
        self.__judul_film = judul_film
        self.__kursi = kursi
        self.__set_harga_dasar(harga_dasar)

    # Setter dengan validasi (Mencegah harga negatif)
    def __set_harga_dasar(self, harga: float):
        if harga < 0:
            raise ValueError("Error: Harga dasar tidak boleh bernilai negatif!")
        self.__harga_dasar = harga

    # Getter menggunakan property decorator (Cara modern Python)
    @property
    def id_tiket(self) -> str:
        return self.__id_tiket

    @property
    def judul_film(self) -> str:
        return self.__judul_film

    @property
    def kursi(self) -> str:
        return self.__kursi

    @property
    def harga_dasar(self) -> float:
        return self.__harga_dasar

    # Abstract Method yang HARUS di-override oleh kelas turunan (Fondasi Polimorfisme)
    @abstractmethod
    def hitung_harga_akhir(self) -> float:
        pass

    @abstractmethod
    def cetak_tiket(self) -> str:
        pass


# 2. POLYMORPHISM PADA KELAS TURUNAN
class TiketReguler(TiketBioskop):
    def hitung_harga_akhir(self) -> float:
        # Tiket reguler tidak ada biaya tambahan
        return self.harga_dasar

    def cetak_tiket(self) -> str:
        return f"[REGULER] {self.judul_film} | Kursi: {self.kursi} | Total: Rp{self.hitung_harga_akhir():,.2f}"


class TiketVIP(TiketBioskop):
    def __init__(self, id_tiket: str, judul_film: str, kursi: str, harga_dasar: float, akses_lounge: bool):
        super().__init__(id_tiket, judul_film, kursi, harga_dasar)
        self.akses_lounge = akses_lounge

    def hitung_harga_akhir(self) -> float:
        # Biaya tambahan 50% untuk VIP
        return self.harga_dasar * 1.5

    def cetak_tiket(self) -> str:
        lounge = "Ya" if self.akses_lounge else "Tidak"
        return f"[VIP] {self.judul_film} | Kursi: {self.kursi} | Akses Lounge: {lounge} | Total: Rp{self.hitung_harga_akhir():,.2f}"


class TiketIMAX(TiketBioskop):
    def hitung_harga_akhir(self) -> float:
        # Biaya tambahan 30% dan pajak tambahan tetap Rp25.000
        return (self.harga_dasar * 1.3) + 25000

    def cetak_tiket(self) -> str:
        return f"[IMAX 3D] {self.judul_film} | Kursi: {self.kursi} | Kacamata 3D Included | Total: Rp{self.hitung_harga_akhir():,.2f}"


# 3. SISTEM MANAJEMEN (Menerapkan objek)
class SistemPemesanan:
    def __init__(self):
        self.__daftar_pesanan: List[TiketBioskop] = []

    def tambah_pesanan(self, tiket: TiketBioskop):
        # Sistem menerima tipe apa saja selama itu adalah turunan dari TiketBioskop
        self.__daftar_pesanan.append(tiket)
        print(f"Berhasil menambahkan pesanan: {tiket.id_tiket}")

    def tampilkan_semua_tiket(self):
        print("\n=== STRUK PEMESANAN TIKET BIOSKOP ===")
        total_pendapatan = 0
        for tiket in self.__daftar_pesanan:
            # Polimorfisme beraksi di sini:
            # Sistem tidak perlu tahu kelas spesifiknya, cukup panggil cetak_tiket()
            print(tiket.cetak_tiket())
            total_pendapatan += tiket.hitung_harga_akhir()

        print("-" * 37)
        print(f"Total Pembayaran: Rp{total_pendapatan:,.2f}")
        print("=====================================\n")


# 4. SIMULASI EKSEKUSI (Driver Code)
if __name__ == "__main__":
    sistem = SistemPemesanan()

    try:
        # Pembuatan objek dengan polimorfisme
        tiket1 = TiketReguler("T-001", "Dune: Part Two", "A12", 50000)
        tiket2 = TiketVIP("T-002", "Interstellar", "V01", 75000, akses_lounge=True)
        tiket3 = TiketIMAX("T-003", "Oppenheimer", "M15", 60000)

        # Contoh validasi Enkapsulasi (Uncomment baris di bawah ini untuk melihat error)
        # tiket_error = TiketReguler("T-004", "Film Error", "B1", -20000)

        sistem.tambah_pesanan(tiket1)
        sistem.tambah_pesanan(tiket2)
        sistem.tambah_pesanan(tiket3)

        sistem.tampilkan_semua_tiket()

    except ValueError as e:
        print(e)

Berhasil menambahkan pesanan: T-001
Berhasil menambahkan pesanan: T-002
Berhasil menambahkan pesanan: T-003

=== STRUK PEMESANAN TIKET BIOSKOP ===
[REGULER] Dune: Part Two | Kursi: A12 | Total: Rp50,000.00
[VIP] Interstellar | Kursi: V01 | Akses Lounge: Ya | Total: Rp112,500.00
[IMAX 3D] Oppenheimer | Kursi: M15 | Kacamata 3D Included | Total: Rp103,000.00
-------------------------------------
Total Pembayaran: Rp265,500.00

